In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [2]:
#rc_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\Assam\assam_rc_2024-11_reduced.json")
rc_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\assam-flood-expenses\assam_district_2024-11_reduced.json")
risk_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Deployment\IDS-DRR-Assam-Risk-Model\RiskScoreModel\data\risk_score_final_district.csv')

C:\Users\saura\AppData\Local\Temp\ipykernel_69608\2366807554.py:3: DtypeWarning: Columns (69,70,71) have mixed types. Specify dtype option on import or set low_memory=False.
  risk_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Deployment\IDS-DRR-Assam-Risk-Model\RiskScoreModel\data\risk_score_final_district.csv')


In [3]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [4]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in rc_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [5]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
rc_gdf['geometry'] = rc_gdf['geometry'].apply(simplify_multipolygon)

geo_fixed = gpd.GeoDataFrame(rc_gdf, geometry='geometry')


In [6]:
rc_gdf

,revenue_ci,revenue_cr,HQ,are_new,dtname,object_id,dtcode11,district_LST_percent_deviation,geometry
0,Sadiya,Sadiya,None,851,TINSUKIA,18-309,18-309,-1.836,"POLYGON ((95.48417 27.24135, 95.50370 27.24863..."
1,Bagribari (Pt),Bagribari (Pt),None,213,DHUBRI,18-301,18-301,4.124,"POLYGON ((89.84312 25.88288, 89.85048 25.87964..."
2,Kalgachia,Kalgachia,None,232,BARPETA,18-303,18-303,1.062,"POLYGON ((91.22210 26.17940, 91.23733 26.18028..."
3,Ujani Majuli,Ujani Majuli,None,322,MAJULI,18-760,18-760,-4.036,"POLYGON ((94.57697 27.16797, 94.57225 27.17411..."
4,Algapur,Algapur,None,156,HAILAKANDI,18-318,18-318,-4.121,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
5,Rangia (Pt),Rangia (Pt),None,23,TAMULPUR,18-816,18-816,0.952,"POLYGON ((91.77912 26.47961, 91.78786 26.47533..."
6,Dimow,Demow,None,478,SIVASAGAR,18-311,18-311,-3.293,"POLYGON ((94.73376 26.93260, 94.73374 26.93261..."
7,Baganpara (Pt),Baganpara (Pt),None,212,BAKSA,18-324,18-324,2.445,"POLYGON ((91.24956 26.52328, 91.26269 26.52165..."
8,Rangia(Pt),Rangia\n(Pt),None,186,KAMRUP,18-321,18-321,0.634,"POLYGON ((91.03041 25.88595, 91.02766 25.86919..."
9,Dhakuakhana (Pt-II),Dhakuakhana (Pt-II),None,9,DHEMAJI,18-308,18-308,-2.893,"POLYGON ((94.26818 27.52094, 94.27895 27.49520..."


In [7]:
merged_gdf = risk_df.merge(geo_fixed[['geometry','object_id']], left_on='object-id', right_on='object_id', how='left')
#merged_gdf = merged_gdf.drop(columns=['dtcode11','block_lgd','dtname','block_name'])
merged_gdf

,object-id,district,rc-area,timeperiod,total-tender-awarded-value,sopd-tenders-awarded-value,sdrf-sanctions-awarded-value,sdrf-tenders-awarded-value,ridf-tenders-awarded-value,ltif-tenders-awarded-value,...,flood-hazard,flood-hazard-float,government-response,financial-year,vulnerability,efficiency,topsis-score,risk-score,geometry,object_id
0,18-758-00257,SOUTH SALMARA MANCACHAR,249,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,5,1.824316,5,2021-2022,4,0.829466,0.889910,5,None,NaN
1,18-756-00249,BISWANATH,1042,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,4,0.863256,5,2021-2022,5,0.784425,0.749157,5,None,NaN
2,18-303-00121,BARPETA,513,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,4,0.825361,5,2021-2022,5,0.769380,0.749157,5,None,NaN
3,18-320-00206,CHIRANG,229,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,5,0.982730,5,2021-2022,4,0.831167,0.744425,5,None,NaN
4,18-316-00185,CACHAR,838,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,4,0.806049,5,2021-2022,3,0.843519,0.732745,4,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9670,18-759,WEST KARBI ANGLONG,3073,2024_08,64956248.00,0.0,0.0,0.0,64956248.0,0.0,...,2,NaN,4,2024-2025,5,0.729127,0.391935,2,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759
9671,18-759,WEST KARBI ANGLONG,3073,2024_09,29350738.42,0.0,0.0,0.0,0.0,0.0,...,1,NaN,4,2024-2025,4,0.805980,0.270089,1,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759
9672,18-759,WEST KARBI ANGLONG,3073,2024_10,0.00,0.0,0.0,0.0,0.0,0.0,...,1,NaN,4,2024-2025,4,0.805980,0.269582,2,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759
9673,18-759,WEST KARBI ANGLONG,3073,2024_11,0.00,0.0,0.0,0.0,0.0,0.0,...,2,NaN,4,2024-2025,4,0.805980,0.362912,3,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759


In [8]:
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)
merged_gdf

,object-id,district,rc-area,timeperiod,total-tender-awarded-value,sopd-tenders-awarded-value,sdrf-sanctions-awarded-value,sdrf-tenders-awarded-value,ridf-tenders-awarded-value,ltif-tenders-awarded-value,...,flood-hazard-float,government-response,financial-year,vulnerability,efficiency,topsis-score,risk-score,geometry,object_id,polygons
0,18-758-00257,SOUTH SALMARA MANCACHAR,249,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,1.824316,5,2021-2022,4,0.829466,0.889910,5,None,NaN,None
1,18-756-00249,BISWANATH,1042,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,0.863256,5,2021-2022,5,0.784425,0.749157,5,None,NaN,None
2,18-303-00121,BARPETA,513,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,0.825361,5,2021-2022,5,0.769380,0.749157,5,None,NaN,None
3,18-320-00206,CHIRANG,229,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,0.982730,5,2021-2022,4,0.831167,0.744425,5,None,NaN,None
4,18-316-00185,CACHAR,838,2021_04,0.00,0.0,0.0,0.0,0.0,0.0,...,0.806049,5,2021-2022,3,0.843519,0.732745,4,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9670,18-759,WEST KARBI ANGLONG,3073,2024_08,64956248.00,0.0,0.0,0.0,64956248.0,0.0,...,NaN,4,2024-2025,5,0.729127,0.391935,2,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759,"[[92.75359088530703, 26.05645072735771], [92.7..."
9671,18-759,WEST KARBI ANGLONG,3073,2024_09,29350738.42,0.0,0.0,0.0,0.0,0.0,...,NaN,4,2024-2025,4,0.805980,0.270089,1,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759,"[[92.75359088530703, 26.05645072735771], [92.7..."
9672,18-759,WEST KARBI ANGLONG,3073,2024_10,0.00,0.0,0.0,0.0,0.0,0.0,...,NaN,4,2024-2025,4,0.805980,0.269582,2,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759,"[[92.75359088530703, 26.05645072735771], [92.7..."
9673,18-759,WEST KARBI ANGLONG,3073,2024_11,0.00,0.0,0.0,0.0,0.0,0.0,...,NaN,4,2024-2025,4,0.805980,0.362912,3,"POLYGON ((92.75359 26.05645, 92.74307 26.06143...",18-759,"[[92.75359088530703, 26.05645072735771], [92.7..."


In [9]:
merged_gdf = merged_gdf.dropna(subset =['object_id'])

In [10]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged_gdf['timeperiod_iso'] = pd.to_datetime(merged_gdf['timeperiod_iso'], format='%Y-%m')


merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod_iso'].dt.strftime('%Y-%m')

C:\Users\saura\AppData\Local\Temp\ipykernel_69608\963400720.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod'].str.replace('_', '-') #+ '-01'
C:\Users\saura\AppData\Local\Temp\ipykernel_69608\963400720.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = pd.to_datetime(merged_gdf['timeperiod_iso'], format='%Y-%m')
C:\Users\saura\AppData\Local\Temp\ipykernel_69608\963400720.py:12: SettingWithCopyWarning: 
A value is trying

In [11]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\Assam\assam_risk.csv')

### Datetime ISO edits

C:\Users\saura\AppData\Local\Temp\ipykernel_42412\963400720.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = merged_gdf['timeperiod'].str.replace('_', '-') #+ '-01'
C:\Users\saura\AppData\Local\Temp\ipykernel_42412\963400720.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_gdf['timeperiod_iso'] = pd.to_datetime(merged_gdf['timeperiod_iso'], format='%Y-%m')
C:\Users\saura\AppData\Local\Temp\ipykernel_42412\963400720.py:12: SettingWithCopyWarning: 
A value is trying

In [ ]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\risk_score_polygons.csv')